# 第二周 · 第 2 天：核心金融指标

> 目标：理解 K 线结构、均线系统、成交量与换手率、市盈率与市净率、ROE 等财务指标

---

## 1. K 线结构（OHLCV）

K 线是技术分析的基础，每个 K 线包含：

| 字段 | 含义 | 说明 |
|------|------|------|
| Open（开盘） | 当天开盘价 | 09:30 成交的第一笔价格 |
| High（最高） | 当天最高价 | 全天最高成交价 |
| Low（最低） | 当天最低价 | 全天最低成交价 |
| Close（收盘） | 当天收盘价 | 15:00 的成交价格 |
| Volume（成交量） | 当天成交量 | 单位：股 |
| Amount（成交额） | 当天成交额 | 单位：元 |

### K 线形态

```python
# 获取 K 线数据
import akshare as ak
import pandas as pd

df = ak.stock_zh_a_hist(symbol="000001", period="daily", start_date="20250601", end_date="20260630")
print(f"列名: {list(df.columns)}")
print(df[['日期', '开盘', '最高', '最低', '收盘', '成交量', '成交额']].head(10))
```

---

## 2. 均线系统（MA）

均线是过去 N 天的收盘价平均值，技术分析中最常用的均线：

| 均线 | 周期 | 应用场景 |
|------|------|----------|
| MA5 | 1 周 | 短期趋势 |
| MA10 | 2 周 | 短期趋势 |
| MA20 | 1 个月 | 中期趋势 |
| MA60 | 3 个月 | 中期趋势 |
| MA120 | 6 个月 | 长期趋势 |
| MA250 | 1 年 | 长期趋势 |

### 计算均线

```python
# 计算均线
df['MA5'] = df['收盘'].rolling(window=5).mean()
df['MA10'] = df['收盘'].rolling(window=10).mean()
df['MA20'] = df['收盘'].rolling(window=20).mean()
df['MA60'] = df['收盘'].rolling(window=60).mean()

print(df[['日期', '收盘', 'MA5', 'MA10', 'MA20', 'MA60']].tail(10))
```

### 均线金叉与死叉

- **金叉**：短期均线上穿长期均线 → 买入信号
  - 例如：MA5 上穿 MA20
- **死叉**：短期均线下穿长期均线 → 卖出信号
  - 例如：MA5 下穿 MA20

```python
# 金叉/死叉判断
df['MA5'] = df['收盘'].rolling(5).mean()
df['MA20'] = df['收盘'].rolling(20).mean()

# 金叉：MA5 从下方穿越 MA20
df['golden_cross'] = (df['MA5'] > df['MA20']) & (df['MA5'].shift(1) <= df['MA20'].shift(1))

# 死叉：MA5 从上方穿越 MA20
df['death_cross'] = (df['MA5'] < df['MA20']) & (df['MA5'].shift(1) >= df['MA20'].shift(1))

print("金叉信号日期:", df[df['golden_cross']]['日期'].tolist())
print("死叉信号日期:", df[df['death_cross']]['日期'].tolist())
```

---

## 3. 成交量与换手率

### 成交量（Volume）

- 反映市场活跃度
- 放量上涨 = 上涨可靠性高
- 缩量下跌 = 可能见底

### 换手率（Turnover Rate）

$$换手率 = \frac{成交量}{流通股本} \times 100\%$$

| 换手率范围 | 市场含义 |
|------------|----------|
| < 3% | 交易清淡，可能盘整 |
| 3% - 10% | 正常交易活跃度 |
| > 10% | 高度活跃，需关注 |
| > 20% | 异常活跃，可能是热点股或主力出货 |

```python
print(df[['日期', '成交量', '成交额', '换手率']].tail(10))

# 放量上涨筛选：涨幅 > 2% 且换手率 > 5%
condition = (df['涨跌幅'] > 2) & (df['换手率'] > 5)
print(f"\n放量上涨次数: {condition.sum()}")
print(df[condition][['日期', '涨跌幅', '换手率']])
```

---

## 4. 市盈率（PE）

$$PE = \frac{股价}{每股收益(EPS)} = \frac{市值}{净利润}$$

| PE 范围 | 估值判断 |
|---------|----------|
| < 10 | 低估值，可能被低估 |
| 10 - 20 | 合理估值 |
| 20 - 40 | 高估值，风险积累 |
| > 40 | 极高估值，需谨慎 |
| 负数 | 亏损状态，无法用 PE 评估 |

> ⚠️ **注意**：PE 适合评估成熟稳定行业，不适合评估亏损公司或强周期行业（券商、煤炭、钢铁等）

```python
# 获取实时行情（含 PE）
df_spot = ak.stock_zh_a_spot_em()

# 筛选低估值股票（PE 5-15，剔除负数）
df_value = df_spot[(df_spot['市盈率-动态'] > 0) & (df_spot['市盈率-动态'] < 15)]
df_value = df_value.sort_values('市盈率-动态')

print(f"低估值股票（PE 0-15）数量: {len(df_value)}")
print(df_value[['代码', '名称', '最新价', '涨跌幅', '市盈率-动态', '市净率']].head(20))
```

---

## 5. 市净率（PB）

$$PB = \frac{股价}{每股净资产(BPS)}$$

| PB 范围 | 估值判断 |
|---------|----------|
| < 1 | 破净股，价格低于净资产 |
| 1 - 3 | 正常范围 |
| 3 - 10 | 高净资产溢价 |
| > 10 | 极高溢价，可能有泡沫 |

> ⚠️ **注意**：PB 适合评估金融股（银行、券商、保险）和强周期行业

```python
# 破净股筛选（PB < 1）
df_pb_low = df_spot[(df_spot['市净率'] > 0) & (df_spot['市净率'] < 1)]
df_pb_low = df_pb_low.sort_values('市净率')

print(f"破净股数量: {len(df_pb_low)}")
print(df_pb_low[['代码', '名称', '最新价', '市净率', '市盈率-动态', '总市值']].head(10))
```

---

## 6. 财务指标综合筛选

### ROE（净资产收益率）

$$ROE = \frac{净利润}{净资产} \times 100\%$$

- 衡量公司盈利能力
- ROE > 15% 为优秀公司

### 营收增速和净利润增速

- 营收增速 > 10% 为成长股
- 净利润增速 > 20% 为高成长股

```python
# 使用 akshare 获取财务数据
# 个股财务指标
df_finance = ak.stock_financial_analysis_indicator(symbol="000001")
print("财务指标列名:", list(df_finance.columns)[:20])
print(df_finance[['日期', '净资产收益率', '销售毛利率', '销售净利率', '资产负债比率']].head(10))
```

---

## 7. 综合选股示例

```python
# 筛选条件：
# 1. PE 在 10-30 之间
# 2. PB 在 1-5 之间
# 3. 换手率 > 2%
# 4. 总市值 > 100 亿（排除小盘股）

market_cap_threshold = 100 * 1e8  # 100 亿

df_filtered = df_spot[
    (df_spot['市盈率-动态'] > 0) &
    (df_spot['市盈率-动态'] < 30) &
    (df_spot['市净率'] > 1) &
    (df_spot['市净率'] < 5) &
    (df_spot['换手率'] > 2) &
    (df_spot['总市值'] > market_cap_threshold)
].copy()

# 按 PE 从低到高排序
df_filtered = df_filtered.sort_values('市盈率-动态')

print(f"符合条件股票数量: {len(df_filtered)}")
print(df_filtered[['代码', '名称', '最新价', '涨跌幅', '市盈率-动态', '市净率', '换手率', '总市值']].head(20))
```

---

## 8. 今日任务清单

- [x] 理解 K 线结构（OHLCV）
- [x] 理解均线系统（MA5/10/20/60）
- [x] 理解金叉与死叉信号
- [x] 理解成交量与换手率含义
- [x] 理解市盈率（PE）估值方法
- [x] 理解市净率（PB）估值方法
- [x] 掌握用 akshare 获取并分析这些指标

---

## ✅ 今日要点总结

| 指标 | 核心含义 | 常用范围/判断 |
|------|----------|---------------|
| K 线 | OHLCV | 技术分析基础 |
| MA5/10/20/60 | 移动平均线 | 金叉买入，死叉卖出 |
| 换手率 | 股票周转频率 | <3%清淡，>10%高度活跃 |
| PE | 估值/回本周期 | 10-20合理，>40高估 |
| PB | 净资产溢价 | 1-3正常，<1破净 |
| ROE | 盈利能力 | >15%优秀 |

---

> 完成日期：2026-07-01
> 下一步：第二周 第 3-4 天 - 财务数据获取（利润表、资产负债表、现金流量表）

In [ ]:
import akshare as ak
import pandas as pd

# 获取 K 线数据
df = ak.stock_zh_a_hist(symbol="000001", period="daily", start_date="20250601", end_date="20260630")
print(f"列名: {list(df.columns)}")
print(df[['日期', '开盘', '最高', '最低', '收盘', '成交量', '成交额']].head(10))

In [ ]:
# 计算均线
df['MA5'] = df['收盘'].rolling(window=5).mean()
df['MA10'] = df['收盘'].rolling(window=10).mean()
df['MA20'] = df['收盘'].rolling(window=20).mean()
df['MA60'] = df['收盘'].rolling(window=60).mean()

print(df[['日期', '收盘', 'MA5', 'MA10', 'MA20', 'MA60']].tail(10))

In [ ]:
# 金叉/死叉判断
df['MA5'] = df['收盘'].rolling(5).mean()
df['MA20'] = df['收盘'].rolling(20).mean()

# 金叉：MA5 从下方穿越 MA20
df['golden_cross'] = (df['MA5'] > df['MA20']) & (df['MA5'].shift(1) <= df['MA20'].shift(1))

# 死叉：MA5 从上方穿越 MA20
df['death_cross'] = (df['MA5'] < df['MA20']) & (df['MA5'].shift(1) >= df['MA20'].shift(1))

print("金叉信号日期:", df[df['golden_cross']]['日期'].tolist())
print("死叉信号日期:", df[df['death_cross']]['日期'].tolist())

In [ ]:
# 换手率分析
print(df[['日期', '成交量', '成交额', '换手率']].tail(10))

# 放量上涨筛选：涨幅 > 2% 且换手率 > 5%
condition = (df['涨跌幅'] > 2) & (df['换手率'] > 5)
print(f"\n放量上涨次数: {condition.sum()}")
print(df[condition][['日期', '涨跌幅', '换手率']])

In [ ]:
# 获取实时行情（含 PE、PB）
df_spot = ak.stock_zh_a_spot_em()

# 筛选低估值股票（PE 5-15，剔除负数）
df_value = df_spot[(df_spot['市盈率-动态'] > 0) & (df_spot['市盈率-动态'] < 15)]
df_value = df_value.sort_values('市盈率-动态')

print(f"低估值股票（PE 0-15）数量: {len(df_value)}")
print(df_value[['代码', '名称', '最新价', '涨跌幅', '市盈率-动态', '市净率']].head(20))

In [ ]:
# 破净股筛选（PB < 1）
df_pb_low = df_spot[(df_spot['市净率'] > 0) & (df_spot['市净率'] < 1)]
df_pb_low = df_pb_low.sort_values('市净率')

print(f"破净股数量: {len(df_pb_low)}")
print(df_pb_low[['代码', '名称', '最新价', '市净率', '市盈率-动态', '总市值']].head(10))

In [ ]:
# 综合选股示例
market_cap_threshold = 100 * 1e8  # 100 亿

df_filtered = df_spot[
    (df_spot['市盈率-动态'] > 0) &
    (df_spot['市盈率-动态'] < 30) &
    (df_spot['市净率'] > 1) &
    (df_spot['市净率'] < 5) &
    (df_spot['换手率'] > 2) &
    (df_spot['总市值'] > market_cap_threshold)
].copy()

# 按 PE 从低到高排序
df_filtered = df_filtered.sort_values('市盈率-动态')

print(f"符合条件股票数量: {len(df_filtered)}")
print(df_filtered[['代码', '名称', '最新价', '涨跌幅', '市盈率-动态', '市净率', '换手率', '总市值']].head(20))